# Data Description
- Background: Unit 1.7 補充資料驗證：全國傷亡事故。提供機關：內政部警政署；官方來源：https://data.gov.tw/dataset/172969 ; https://data.gov.tw/dataset/177136。原始來源對應與官方最新資源bytes完全相同為不同命題；除歷史界內包已逐byte雜湊對應外，未重新下載認證其他最新資源。
- date received: 學生於本單元提供；實際下載／收到日期未提供。首次本機檢視2026-11-26，不將今日當下載日。
- Path to data file: 官方來源及版本如上；精確本機容器如下，解壓至data/temp僅為衍生暫存。
  - `data/raw/113年傷亡道路交通事故資料.zip`；40,598,237 bytes；SHA-256 `f5a8802333ebbd5ba6573e3d7d14e3d8fb1e57be238272140e39d07c66a89e63`。
  - `data/raw/114年傷亡道路交通事故資料.zip`；43,253,520 bytes；SHA-256 `2a8e32266708fba9bd9764cefc07717ded978f65bae834c3b6795a992d04fbc7`。
- Unit of observation: 全國每列當事人；本次分批篩臺北地址及順位1，處理警局口徑另報。
- Sample period: 事故發生2024／2025；A1全年＋A2逐月，實際月份由檔內驗證；發布／更新：113資料集2025-03-03；114資料集2026-03-06；metadata更新2026-08。ZIP時間、檔名時間、觀測／設置時間分開，不由檔名猜全年存量。
- Known issues: 每列當事人，先臺北篩選再順位1；不同公開版本不能附加重複事件或當獨立位置認證。缺失、重複、坐標及CRS由下方實讀輸出；沒有完整性證據的0僅為本檔未觀測。授權：政府資料開放授權條款第1版；依官方頁，來源與版本須保留。
- Definition for each variable: 下方逐欄型態、缺失、觀測域與定義表。已知官方欄意與本次診斷欄分開；未獲欄位碼表支持者明記未知，不依名稱猜設施單位。
- Data path: 上列每個獨立原始容器；實際讀取成員列於Summary Stats。
- Role: raw。
- Verification status: 以本Notebook乾淨kernel實際輸出與執行metadata為準；不認證已清理。
- SHA-256: 上列每檔固定綁定，載入前逐項重核。
- Next processing notebook: 本單元僅診斷，Chapter 2尚未建立；[驗證一](../code/taipei_supplementary_accident_boundary_validation.ipynb)、[驗證二](../code/taipei_transportation_infrastructure_validation.ipynb)。

Author: 詹明翰、謝子詮、曾翊涵；Created At / Last Modified At: 2026-11-26。

Primary Owner：謝子詮（DE）；Collaborator / Analytical Reviewer：詹明翰（DA）；Project Reviewer：曾翊涵（PM）。共同作者與職務不代表已完成個別工作。本輪由Codex協助執行，成員驗收與審查尚待完成。


In [1]:
from pathlib import Path
import sys, hashlib, json, zipfile
import pandas as pd
from IPython.display import display, Markdown
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate/'AGENTS.md').is_file() and (candidate/'templates').is_dir():
        PROJECT_ROOT=candidate;break
else:raise FileNotFoundError('請從本研究工作包內執行')
def fingerprint(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()
assert fingerprint(PROJECT_ROOT/'data/code/unit17_validation_config.json')=='7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7'
assert fingerprint(PROJECT_ROOT/'data/code/unit17_validation_support.py')=='1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080'
sys.path.insert(0,str(PROJECT_ROOT/'data/code'))
from unit17_validation_support import Context, show, safe_text
pd.set_option('display.max_columns',None)
pd.set_option('display.max_colwidth',160)
ctx=Context(PROJECT_ROOT,'7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7')
print('Python:',sys.version,'Config:','7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7','Helper:','1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080')


Python: 3.12.2 (tags/v3.12.2:6abddd9, Feb  6 2024, 21:26:36) [MSC v.1937 64 bit (AMD64)] Config: 7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7 Helper: 1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080


## Change for Your Own Data

使用已核對精確路徑及固定SHA-256；不選第一個glob或最新檔。CP950非法bytes只作可逆診斷；大檔分批讀取、OSM僅研究區2km bbox，保留全國母體數。

In [2]:
FAMILY='national_accidents'
INPUTS=ctx.family_items(FAMILY)
for item in INPUTS:ctx.file(item['path'])
frames=ctx.load_family(FAMILY)

## Summary

每個容器、實際成員與載入範圍分列；沒有將暫存解壓檔當原始下載資料。

In [3]:
show('固定輸入身分',pd.DataFrame([{k:v for k,v in r.items() if k not in ['members','encoding_checks']} for r in INPUTS]))
show('讀取範圍與CRS／編碼',pd.DataFrame(ctx.summaries[FAMILY]))

**固定輸入身分**

,family,path,bytes,sha256,status
0,national_accidents,data/raw/113年傷亡道路交通事故資料.zip,40598237,f5a8802333ebbd5ba6573e3d7d14e3d8fb1e57be238272140e39d07c66a89e63,present
1,national_accidents,data/raw/114年傷亡道路交通事故資料.zip,43253520,2a8e32266708fba9bd9764cefc07717ded978f65bae834c3b6795a992d04fbc7,present


**讀取範圍與CRS／編碼**

,path,member,encoding,schema,national_raw_csv_rows,national_party_rows,non_observation_rows,footer_or_non_observation_records,taipei_address_party_rows,taipei_address_first,taipei_agency_first,observed_months
0,data/raw/113年傷亡道路交通事故資料.zip,113年度A1交通事故資料.csv,utf-8-sig,"[發生年度, 發生月份, 發生日期, 發生時間, 事故類別名稱, 處理單位名稱警局層, 發生地點, 天候名稱, 光線名稱, 道路類別-第1當事者-名稱, 速限-第1當事者, 道路型態大類別名稱, 道路型態子類別名稱, 事故位置大類別名稱, 事故位置子類別名稱, 路面狀況-路面鋪裝名稱, 路面狀況-路面狀態名稱,...",4372,4370,2,"[{'source_row': 4371, 'year_field': '資料提供日期：114年07月01日', 'date_field': '', 'category_field': ''}, {'source_row': 4372, 'year_field': '事故類別：A1類', 'date_field...",215,71,68,"{'10': 408, '2': 407, '12': 402, '8': 397, '1': 395, '7': 393, '3': 380, '11': 356, '6': 330, '9': 315, '4': 296, '5': 291}"
1,data/raw/113年傷亡道路交通事故資料.zip,113年度A2交通事故資料_1.csv,utf-8-sig,"[發生年度, 發生月份, 發生日期, 發生時間, 事故類別名稱, 處理單位名稱警局層, 發生地點, 天候名稱, 光線名稱, 道路類別-第1當事者-名稱, 速限-第1當事者, 道路型態大類別名稱, 道路型態子類別名稱, 事故位置大類別名稱, 事故位置子類別名稱, 路面狀況-路面鋪裝名稱, 路面狀況-路面狀態名稱,...",80426,80424,2,"[{'source_row': 80425, 'year_field': '資料提供日期：114年07月01日', 'date_field': '', 'category_field': ''}, {'source_row': 80426, 'year_field': '事故類別：A2類', 'date_fie...",4701,2051,2043,{'1': 80424}
2,data/raw/113年傷亡道路交通事故資料.zip,113年度A2交通事故資料_10.csv,utf-8-sig,"[發生年度, 發生月份, 發生日期, 發生時間, 事故類別名稱, 處理單位名稱警局層, 發生地點, 天候名稱, 光線名稱, 道路類別-第1當事者-名稱, 速限-第1當事者, 道路型態大類別名稱, 道路型態子類別名稱, 事故位置大類別名稱, 事故位置子類別名稱, 路面狀況-路面鋪裝名稱, 路面狀況-路面狀態名稱,...",70341,70339,2,"[{'source_row': 70340, 'year_field': '資料提供日期：114年07月01日', 'date_field': '', 'category_field': ''}, {'source_row': 70341, 'year_field': '事故類別：A2類', 'date_fie...",3904,1726,1710,{'10': 70339}
3,data/raw/113年傷亡道路交通事故資料.zip,113年度A2交通事故資料_11.csv,utf-8-sig,"[發生年度, 發生月份, 發生日期, 發生時間, 事故類別名稱, 處理單位名稱警局層, 發生地點, 天候名稱, 光線名稱, 道路類別-第1當事者-名稱, 速限-第1當事者, 道路型態大類別名稱, 道路型態子類別名稱, 事故位置大類別名稱, 事故位置子類別名稱, 路面狀況-路面鋪裝名稱, 路面狀況-路面狀態名稱,...",76692,76690,2,"[{'source_row': 76691, 'year_field': '資料提供日期：114年07月01日', 'date_field': '', 'category_field': ''}, {'source_row': 76692, 'year_field': '事故類別：A2類', 'date_fie...",4197,1811,1788,{'11': 76690}
4,data/raw/113年傷亡道路交通事故資料.zip,113年度A2交通事故資料_12.csv,utf-8-sig,"[發生年度, 發生月份, 發生日期, 發生時間, 事故類別名稱, 處理單位名稱警局層, 發生地點, 天候名稱, 光線名稱, 道路類別-第1當事者-名稱, 速限-第1當事者, 道路型態大類別名稱, 道路型態子類別名稱, 事故位置大類別名稱, 事故位置子類別名稱, 路面狀況-路面鋪裝名稱, 路面狀況-路面狀態名稱,...",84558,84556,2,"[{'source_row': 84557, 'year_field': '資料提供日期：114年07月01日', 'date_field': '', 'category_field': ''}, {'source_row': 84558, 'year_field': '事故類別：A2類', 'date_fie...",4656,1991,1977,{'12': 84556}
5,data/raw/113年傷亡道路交通事故資料.zip,113年度A2交通事故資料_2.csv,utf-8-sig,"[發生年度, 發生月份, 發生日期, 發生時間, 事故類別名稱, 處理單位名稱警局層, 發生地點, 天候名稱, 光線名稱, 道路類別-第1當事者-名稱, 速限-第1當事者, 道路型態大類別名稱, 道路型態子類別名稱, 事故位置大類別名稱, 事故位置子類別名稱, 路面狀況-路面鋪裝名稱, 路面狀況-路面狀態名稱,...",66755,66753,2,"[{'source_row': 66754, 'year_field': '資料提供日期：114年07月01日', 'date_field': '', 'category_field': ''}, {'source_row': 66755, 'year_field': '事故類別：A2類', 'date_fie...",3678,1574,1559,{'2': 66753}
6,data/raw/113年傷亡道路交通事故資料.zip,113年度A2交通事故資料_3.csv,utf-8-sig,"[發生年度, 發生月份, 發生日期, 發生時間, 事故類別名稱, 處理單位名稱警局層, 發生地點, 天候名稱, 光線名稱, 道路類別-第1當事者-名稱, 速限-第1當事者, 道路型態大類別名稱, 道路型態子類別名稱, 事故位置大類別名稱, 事故位置子類別名稱, 路面狀況-路面鋪裝名稱, 路面狀況-路面狀態名稱,...",72236,72234,2,"[{'source_row': 72235, 'year_field': '資料提供日期：114年07月01日', 'date_field': '', 'category_field': ''}, {'source_row': 72236, 'year_field': '事故類別：A2類', 'date_fie...",4416,1886,1863,{'3': 72234}
7,data/raw/113年傷亡道路交通事故資料.zip,113年度A2交通事故資料_4.csv,utf-8-sig,"[發生年度, 發生月份, 發生日期, 發生時間, 事故類別名稱, 處理單位名稱警局層, 發生地點, 天候名稱, 光線名稱, 道路類別-第1當事者-名稱, 速限-第1當事者, 道路型態大類別名稱, 道路型態子類別名稱, 事故位置大類別名稱, 事故位置子類別名稱, 路面狀況-路面鋪裝名稱, 路面狀況-路面狀態名稱,...",68747,68745,2,"[{'source_row': 68746, 'year_field': '資料提供日期：114年07月01日', 'date_field': '', 'category_field': ''}, {'source_row': 68747, 'year_field': '事故類別：A2類', 'date_fie...",4054,1747,1727,{'4': 68745}
8,data/raw/113年傷亡道路交通事故資料.zip,113年度A2交通事故資料_5.csv,utf-8-sig,"[發生年度, 發生月份, 發生日期, 發生時間, 事故類別名稱, 處理單位名稱警局層, 發生地點, 天候名稱, 光線名稱, 道路類別-第1當事者-名稱, 速限-第1當事者, 道路型態大類別名稱, 道路型態子類別名稱, 事故位置大類別名稱, 事故位置子類別名稱, 路面狀況-路面鋪裝名稱, 路面狀況-路面狀態名稱,...",72537,72535,2,"[{'source_row': 72536, 'year_field':

,path,member,encoding,schema,national_raw_csv_rows,national_party_rows,non_observation_rows,footer_or_non_observation_records,taipei_address_party_rows,taipei_address_first,taipei_agency_first,observed_months
18,data/raw/114年傷亡道路交通事故資料.zip,114年度A2交通事故資料_2.csv,utf-8-sig,"[發生年度, 發生月份, 發生日期, 發生時間, 事故類別名稱, 處理單位名稱警局層, 發生地點, 天候名稱, 光線名稱, 道路類別-第1當事者-名稱, 速限-第1當事者, 道路型態大類別名稱, 道路型態子類別名稱, 事故位置大類別名稱, 事故位置子類別名稱, 路面狀況-路面鋪裝名稱, 路面狀況-路面狀態名稱,...",66239,66237,2,"[{'source_row': 66238, 'year_field': '資料提供日期：115年07月01日', 'date_field': '', 'category_field': ''}, {'source_row': 66239, 'year_field': '事故類別：A2類', 'date_fie...",3760,1632,1621,{'2': 66237}
19,data/raw/114年傷亡道路交通事故資料.zip,114年度A2交通事故資料_3.csv,utf-8-sig,"[發生年度, 發生月份, 發生日期, 發生時間, 事故類別名稱, 處理單位名稱警局層, 發生地點, 天候名稱, 光線名稱, 道路類別-第1當事者-名稱, 速限-第1當事者, 道路型態大類別名稱, 道路型態子類別名稱, 事故位置大類別名稱, 事故位置子類別名稱, 路面狀況-路面鋪裝名稱, 路面狀況-路面狀態名稱,...",76106,76104,2,"[{'source_row': 76105, 'year_field': '資料提供日期：115年07月01日', 'date_field': '', 'category_field': ''}, {'source_row': 76106, 'year_field': '事故類別：A2類', 'date_fie...",4688,2007,1988,{'3': 76104}
20,data/raw/114年傷亡道路交通事故資料.zip,114年度A2交通事故資料_4.csv,utf-8-sig,"[發生年度, 發生月份, 發生日期, 發生時間, 事故類別名稱, 處理單位名稱警局層, 發生地點, 天候名稱, 光線名稱, 道路類別-第1當事者-名稱, 速限-第1當事者, 道路型態大類別名稱, 道路型態子類別名稱, 事故位置大類別名稱, 事故位置子類別名稱, 路面狀況-路面鋪裝名稱, 路面狀況-路面狀態名稱,...",73063,73061,2,"[{'source_row': 73062, 'year_field': '資料提供日期：115年07月01日', 'date_field': '', 'category_field': ''}, {'source_row': 73063, 'year_field': '事故類別：A2類', 'date_fie...",4296,1848,1830,{'4': 73061}
21,data/raw/114年傷亡道路交通事故資料.zip,114年度A2交通事故資料_5.csv,utf-8-sig,"[發生年度, 發生月份, 發生日期, 發生時間, 事故類別名稱, 處理單位名稱警局層, 發生地點, 天候名稱, 光線名稱, 道路類別-第1當事者-名稱, 速限-第1當事者, 道路型態大類別名稱, 道路型態子類別名稱, 事故位置大類別名稱, 事故位置子類別名稱, 路面狀況-路面鋪裝名稱, 路面狀況-路面狀態名稱,...",74954,74952,2,"[{'source_row': 74953, 'year_field': '資料提供日期：115年07月01日', 'date_field': '', 'category_field': ''}, {'source_row': 74954, 'year_field': '事故類別：A2類', 'date_fie...",4344,1874,1850,{'5': 74952}
22,data/raw/114年傷亡道路交通事故資料.zip,114年度A2交通事故資料_6.csv,utf-8-sig,"[發生年度, 發生月份, 發生日期, 發生時間, 事故類別名稱, 處理單位名稱警局層, 發生地點, 天候名稱, 光線名稱, 道路類別-第1當事者-名稱, 速限-第1當事者, 道路型態大類別名稱, 道路型態子類別名稱, 事故位置大類別名稱, 事故位置子類別名稱, 路面狀況-路面鋪裝名稱, 路面狀況-路面狀態名稱,...",73674,73672,2,"[{'source_row': 73673, 'year_field': '資料提供日期：115年07月01日', 'date_field': '', 'category_field': ''}, {'source_row': 73674, 'year_field': '事故類別：A2類', 'date_fie...",4148,1779,1760,{'6': 73672}
23,data/raw/114年傷亡道路交通事故資料.zip,114年度A2交通事故資料_7.csv,utf-8-sig,"[發生年度, 發生月份, 發生日期, 發生時間, 事故類別名稱, 處理單位名稱警局層, 發生地點, 天候名稱, 光線名稱, 道路類別-第1當事者-名稱, 速限-第1當事者, 道路型態大類別名稱, 道路型態子類別名稱, 事故位置大類別名稱, 事故位置子類別名稱, 路面狀況-路面鋪裝名稱, 路面狀況-路面狀態名稱,...",71586,71584,2,"[{'source_row': 71585, 'year_field': '資料提供日期：115年07月01日', 'date_field': '', 'category_field': ''}, {'source_row': 71586, 'year_field': '事故類別：A2類', 'date_fie...",4269,1853,1833,{'7': 71584}
24,data/raw/114年傷亡道路交通事故資料.zip,114年度A2交通事故資料_8.csv,utf-8-sig,"[發生年度, 發生月份, 發生日期, 發生時間, 事故類別名稱, 處理單位名稱警局層, 發生地點, 天候名稱, 光線名稱, 道路類別-第1當事者-名稱, 速限-第1當事者, 道路型態大類別名稱, 道路型態子類別名稱, 事故位置大類別名稱, 事故位置子類別名稱, 路面狀況-路面鋪裝名稱, 路面狀況-路面狀態名稱,...",74139,74137,2,"[{'source_row': 74138, 'year_field': '資料提供日期：115年07月01日', 'date_field': '', 'category_field': ''}, {'source_row': 74139, 'year_field': '事故類別：A2類', 'date_fie...",4535,1962,1940,{'8': 74137}
25,data/raw/114年傷亡道路交通事故資料.zip,114年度A2交通事故資料_9.csv,utf-8-sig,"[發生年度, 發生月份, 發生日期, 發生時間, 事故類別名稱, 處理單位名稱警局層, 發生地點, 天候名稱, 光線名稱, 道路類別-第1當事者-名稱, 速限-第1當事者, 道路型態大類別名稱, 道路型態子類別名稱, 事故位置大類別名稱, 事故位置子類別名稱, 路面狀況-路面鋪裝名稱, 路面狀況-路面狀態名稱,...",73457,73455,2,"[{'source_row': 73456, 'year_field': '資料提供日期：115年07月01日', 'date_field': '', 'category_field': ''}, {'source_row': 73457, 'year_field': '事故類別：A2類', 'date_fie...",4403,1926,1918,{'9': 73455}


### Sample

各實際讀取表前十列；全國事故為臺北篩選後第一當事人，不代表全國前十列。OSM為bbox空間子集。

In [4]:
for source,frame in frames.items():
    show(source,frame.drop(columns='geometry',errors='ignore').head(10),block=10)

**data/raw/113年傷亡道路交通事故資料.zip**

,發生年度,發生月份,發生日期,發生時間,事故類別名稱,處理單位名稱警局層,發生地點,光線名稱,事故位置大類別名稱,事故位置子類別名稱,事故類型及型態大類別名稱,事故類型及型態子類別名稱,死亡受傷人數,當事者順位,當事者區分-類別-大類別名稱-車種,當事者區分-類別-子類別名稱-車種,經度,緯度,_source_row,_source_member,_source_path,_city_address,_taipei_agency
0,2024,1,20240108,110400,A1,國道公路警察局,臺北市南港區國道3號 16公里300.0公尺處北側向外側,有照明未開啟或故障,路段,快車道,車與車,追撞,死亡1;受傷8,1,半聯結車,營業用,121.598520,25.018276,77,113年度A1交通事故資料.csv,data/raw/113年傷亡道路交通事故資料.zip,True,False
1,2024,1,20240119,091400,A1,臺北市政府警察局,臺北市中山區八德路2段 / 臺北市中山區龍江路,無照明,交叉路口,交叉口附近,車與車,追撞,死亡1;受傷0,1,機車,普通重型,121.540162,25.046653,242,113年度A1交通事故資料.csv,data/raw/113年傷亡道路交通事故資料.zip,True,True
2,2024,1,20240121,063000,A1,臺北市政府警察局,臺北市士林區福華路 / 臺北市士林區福華路,有照明未開啟或故障,交叉路口,交叉口附近,人與車,穿越道路中,死亡1;受傷0,1,小客車(含客、貨兩用),自用,121.522169,25.103749,273,113年度A1交通事故資料.csv,data/raw/113年傷亡道路交通事故資料.zip,True,True
3,2024,1,20240122,194600,A1,臺北市政府警察局,臺北市士林區忠誠路1段 / 臺北市士林區德行東路,有照明且開啟,交叉路口,交叉路口內,人與車,穿越道路中,死亡1;受傷1,1,機車,普通重型,121.529729,25.106804,291,113年度A1交通事故資料.csv,data/raw/113年傷亡道路交通事故資料.zip,True,True
4,2024,1,20240123,023500,A1,臺北市政府警察局,臺北市萬華區環河快速道路前0.0公尺,有照明且開啟,路段,快車道,車輛本身,撞橋樑(橋墩),死亡1;受傷1,1,小客車(含客、貨兩用),自用,121.494906,25.038283,295,113年度A1交通事故資料.csv,data/raw/113年傷亡道路交通事故資料.zip,True,True
5,2024,1,20240126,080600,A1,臺北市政府警察局,臺北市中山區民生東路1段 / 臺北市中山區林森北路,有照明未開啟或故障,交叉路口,交叉路口內,車與車,側撞,死亡1;受傷0,1,小客車(含客、貨兩用),計程車,121.525536,25.058082,326,113年度A1交通事故資料.csv,data/raw/113年傷亡道路交通事故資料.zip,True,True
6,2024,2,20240202,143600,A1,臺北市政府警察局,臺北市北投區立農街1段 / 臺北市北投區吉利街,無照明,交叉路口,交叉路口內,車與車,側撞,死亡1;受傷0,1,小客車(含客、貨兩用),自用,121.506631,25.119081,414,113年度A1交通事故資料.csv,data/raw/113年傷亡道路交通事故資料.zip,True,True
7,2024,2,20240203,134200,A1,臺北市政府警察局,臺北市松山區八德路4段 / 臺北市松山區塔悠路,無照明,交叉路口,交叉路口內,車與車,其他,死亡1;受傷0,1,大客車,營業用,121.572054,25.050292,433,113年度A1交通事故資料.csv,data/raw/113年傷亡道路交通事故資料.zip,True,True
8,2024,2,20240206,141900,A1,臺北市政府警察局,臺北市內湖區行忠路 / 臺北市內湖區行善路,有照明且開啟,交叉路口,交叉路口內,車與車,側撞,死亡1;受傷0,1,小客車(含客、貨兩用),自用,121.585684,25.066320,479,113年度A1交通事故資料.csv,data/raw/113年傷亡道路交通事故資料.zip,True,True
9,2024,2,20240206,151500,A1,臺北市政府警察局,臺北市內湖區堤外便道前0.0公尺,有照明未開啟或故障,路段,一般車道(未劃分快慢車道),車輛本身,撞護欄(樁),死亡1;受傷0,1,機車,普通重型,121.613126,25.065284,481,113年度A1交通事故資料.csv,data/raw/113年傷亡道路交通事故資料.zip,True,True


**data/raw/114年傷亡道路交通事故資料.zip**

,發生年度,發生月份,發生日期,發生時間,事故類別名稱,處理單位名稱警局層,發生地點,光線名稱,事故位置大類別名稱,事故位置子類別名稱,事故類型及型態大類別名稱,事故類型及型態子類別名稱,死亡受傷人數,當事者順位,當事者區分-類別-大類別名稱-車種,當事者區分-類別-子類別名稱-車種,經度,緯度,_source_row,_source_member,_source_path,_city_address,_taipei_agency
0,2025,1,20250121,104400,A1,臺北市政府警察局,臺北市士林區通河西街1段前0.0公尺,無照明,路段,一般車道(未劃分快慢車道),車與車,其他,死亡1;受傷0,1,機車,普通重型,121.515108,25.079969,207,114年度A1交通事故資料.csv,data/raw/114年傷亡道路交通事故資料.zip,True,True
1,2025,1,20250125,033200,A1,臺北市政府警察局,臺北市萬華區萬大路 / 臺北市萬華區環河南路3段,有照明且開啟,交叉路口,交叉口附近,人與車,其他,死亡1;受傷1,1,機車,普通重型,121.495915,25.016916,277,114年度A1交通事故資料.csv,data/raw/114年傷亡道路交通事故資料.zip,True,True
2,2025,2,20250207,215300,A1,臺北市政府警察局,臺北市萬華區華中橋前0.0公尺,有照明且開啟,路段,機車專用道,人與車,同向通行中,死亡1;受傷1,1,機車,普通重型,121.496045,25.017208,429,114年度A1交通事故資料.csv,data/raw/114年傷亡道路交通事故資料.zip,True,True
3,2025,2,20250208,062800,A1,臺北市政府警察局,臺北市松山區基隆路1段 / 臺北市松山區八德路4段,無照明,路段,快車道,人與車,穿越道路中,死亡1;受傷0,1,人,行人,121.569542,25.049656,431,114年度A1交通事故資料.csv,data/raw/114年傷亡道路交通事故資料.zip,True,True
4,2025,2,20250215,084300,A1,臺北市政府警察局,臺北市中山區松江路 / 臺北市中山區民族東路,有照明未開啟或故障,交叉路口,交叉路口內,車與車,側撞,死亡1;受傷0,1,半聯結車,營業用,121.532974,25.068327,486,114年度A1交通事故資料.csv,data/raw/114年傷亡道路交通事故資料.zip,True,True
5,2025,2,20250224,165400,A1,臺北市政府警察局,臺北市內湖區潭美街 / 臺北市內湖區安美街,有照明未開啟或故障,交叉路口,交叉口附近,車與車,追撞,死亡1;受傷1,1,機車,普通重型,121.601248,25.061012,624,114年度A1交通事故資料.csv,data/raw/114年傷亡道路交通事故資料.zip,True,True
6,2025,2,20250227,164800,A1,臺北市政府警察局,臺北市萬華區青年路 / 臺北市萬華區水源路,有照明未開啟或故障,交叉路口,交叉口附近,車與車,對撞,死亡1;受傷0,1,小貨車,自用,121.503984,25.019786,653,114年度A1交通事故資料.csv,data/raw/114年傷亡道路交通事故資料.zip,True,True
7,2025,3,20250304,172700,A1,臺北市政府警察局,臺北市大同區重慶北路3段 / 臺北市大同區昌吉街,有照明且開啟,交叉路口,交叉口附近,人與車,穿越道路中,死亡1;受傷0,1,人,行人,121.513763,25.065839,726,114年度A1交通事故資料.csv,data/raw/114年傷亡道路交通事故資料.zip,True,True
8,2025,3,20250306,015000,A1,臺北市政府警察局,臺北市中正區國興路 / 臺北市中正區中華路2段,有照明且開啟,交叉路口,交叉路口內,車與車,路口交岔撞,死亡1;受傷0,1,慢車,腳踏自行車,121.509215,25.024933,740,114年度A1交通事故資料.csv,data/raw/114年傷亡道路交通事故資料.zip,True,True
9,2025,3,20250311,052400,A1,臺北市政府警察局,臺北市中山區民族東路 / 臺北市中山區新生北路3段,有照明且開啟,交叉路口,交叉路口內,車輛本身,撞交通島,死亡1;受傷0,1,機車,普通重型,121.528164,25.068326,787,114年度A1交通事故資料.csv,data/raw/114年傷亡道路交通事故資料.zip,True,True


### Summary Stats

ZIP逐成員CRC、配套、原始WKT、全部欄型態／缺失／定義、重複與數值／類別／時間診斷。保留原值及bytes。

In [5]:
_ = ctx.raw_report(FAMILY,crc=True)

,source_metadata_and_limits
title,全國傷亡事故
provider,內政部警政署
url,https://data.gov.tw/dataset/172969 ; https://data.gov.tw/dataset/177136
publication,113資料集2025-03-03；114資料集2026-03-06；metadata更新2026-08
period,事故發生2024／2025；A1全年＋A2逐月，實際月份由檔內驗證
limitation,每列當事人，先臺北篩選再順位1；不同公開版本不能附加重複事件或當獨立位置認證
license,政府資料開放授權條款第1版；依官方頁，來源與版本須保留


,family,path,bytes,sha256,status
0,national_accidents,data/raw/113年傷亡道路交通事故資料.zip,40598237,f5a8802333ebbd5ba6573e3d7d14e3d8fb1e57be238272140e39d07c66a89e63,present


**ZIP原始成員清單；非解壓暫存資料來源**

,member,bytes,compressed,crc32,zip_datetime
0,113年度A1交通事故資料.csv,2758536,200063,8e3ce3f2,"[2025, 7, 1, 3, 33, 58]"
1,113年度A2交通事故資料_1.csv,51106105,3667430,67676712,"[2025, 7, 1, 3, 35, 52]"
2,113年度A2交通事故資料_10.csv,44717771,3226488,edb4709b,"[2025, 7, 1, 3, 37, 32]"
3,113年度A2交通事故資料_11.csv,48771828,3515439,5dccaae3,"[2025, 7, 1, 3, 39, 22]"
4,113年度A2交通事故資料_12.csv,53830421,3853226,3c36156b,"[2025, 7, 1, 3, 41, 24]"
5,113年度A2交通事故資料_2.csv,42399040,3039959,09ae872b,"[2025, 7, 1, 3, 43, 0]"
6,113年度A2交通事故資料_3.csv,45854217,3296591,05ea9a04,"[2025, 7, 1, 3, 44, 38]"
7,113年度A2交通事故資料_4.csv,43659909,3141554,3a0c79c7,"[2025, 7, 1, 3, 46, 14]"
8,113年度A2交通事故資料_5.csv,46068215,3307626,872d7821,"[2025, 7, 1, 3, 47, 56]"
9,113年度A2交通事故資料_6.csv,46464008,3349295,33fd3821,"[2025, 7, 1, 3, 49, 38]"


ZIP 全成員CRC通過: data/raw/113年傷亡道路交通事故資料.zip


,family,path,bytes,sha256,status
0,national_accidents,data/raw/114年傷亡道路交通事故資料.zip,43253520,2a8e32266708fba9bd9764cefc07717ded978f65bae834c3b6795a992d04fbc7,present


... (38 outputs left out)

,field,read_for_taipei_diagnostic,definition
36,當事者屬-性-別名稱,False,原始欄名；本次不選讀，精確語義／型態與缺失未驗證
37,當事者事故發生時年齡,False,原始欄名；本次不選讀，精確語義／型態與缺失未驗證
38,保護裝備名稱,False,原始欄名；本次不選讀，精確語義／型態與缺失未驗證
39,行動電話或電腦或其他相類功能裝置名稱,False,原始欄名；本次不選讀，精確語義／型態與缺失未驗證
40,當事者行動狀態大類別名稱,False,原始欄名；本次不選讀，精確語義／型態與缺失未驗證
41,當事者行動狀態子類別名稱,False,原始欄名；本次不選讀，精確語義／型態與缺失未驗證
42,車輛撞擊部位大類別名稱-最初,False,原始欄名；本次不選讀，精確語義／型態與缺失未驗證
43,車輛撞擊部位子類別名稱-最初,False,原始欄名；本次不選讀，精確語義／型態與缺失未驗證
44,車輛撞擊部位大類別名稱-其他,False,原始欄名；本次不選讀，精確語義／型態與缺失未驗證
45,車輛撞擊部位子類別名稱-其他,False,原始欄名；本次不選讀，精確語義／型態與缺失未驗證


原始屬性完整列重複（不含診斷追溯欄） 0


**類別前12項：事故類別名稱**

,事故類別名稱,N,ratio_of_loaded_rows
0,A2,22802,0.997463
1,A1,58,0.002537


本文件只確認資料身分及可讀性；語義／日期不足者限制用途，不強行納入正式Dataset B3。人類驗收仍待完成。

In [6]:
for item in INPUTS:ctx.file(item['path'])
print('原始容器hash重核通過；正式分析資料輸出0。')

原始容器hash重核通過；正式分析資料輸出0。
